# DeepSeek-OCR × CUAD：5× / 10× 标准压缩比对比实验

## 上一轮 notebook 再次出现的 abstention-floor 陷阱（诊断）

`DeepSeek_OCR_CUAD_Colab.ipynb` 得到七种设置 EM 都落在 **0.633**；本 notebook 首轮则出现了更极端的退化——所有方法（包括 `full` 完整上下文基线）`EM_ans=0, F1_ans=0, abstain_acc=1.0`。
两次表象不同，但根因同源：**QA reader 对所有问题都返回了空串**。
- 上一轮：reader 对不可答题系统弃答，可答/不可答比例失衡；
- 本轮：CUAD 原文最长达 36k tokens，超过 `qwen-plus` 的 32k 上下文上限，API 大量 400/超时；同时旧的 `try/except` 把异常静默折叠成空串，让 pipeline 看起来像一次「完美弃答」。

### 本 notebook 的修正方案（保留 5×/10× 三路压缩的评测目标不变）

1. **reader 换成长上下文模型**：默认 `qwen-long`（千万级 token），避免 `full` 基线因长度溢出；
2. **失败不再静默**：每条 run 记录 `status ∈ {ok, api_error, parse_error}` 和 `error_msg`；
3. **预检单元格**：跑 556 次之前先对 `test_items[0]` 的 `full` 上下文做一次调通；
4. **度量统一**：OCR 分支的实际压缩比改用「解码文本的 RoBERTa tokens」作为分母（与 prune/summary 同尺子），视觉 token 数单独保留为辅助列；
5. **评测分母只数成功样本**：`EM_ans / F1_ans / abstain_acc_clean` 只在 `status=='ok'` 上计算，并新增 `success_rate` 列暴露接口健康度。

> 评测协议（分层采样、三路压缩、5×/10× 目标、reader 共享、JSON 置信度）与上一版完全一致；本次修改只**加固测量**，不改变「比什么」。


## 0. 环境准备

硬件：A100-40G / A10G / L4；T4 不支持；MacBook 不支持。

**API keys**（至少一个）：`DEEPSEEK_API_KEY`, `DASHSCOPE_API_KEY`, `OPENAI_API_KEY`。下一单元格按以下顺序加载，先命中者生效：
1. **Colab Secrets**（Runtime → Secrets）——云端推荐；
2. **`notebooks/.env`** 文件（格式 `KEY=VALUE` 每行一个，注释以 `#` 开头）——本地推荐；`.env` 已被 `.gitignore` 忽略，不会进仓库；
3. 已经 `export` 到 shell 的环境变量。


In [1]:
import os
from pathlib import Path

# Load API keys from three sources, in priority order (first wins):
#   1. Colab Secrets (userdata) — if running on Colab with Secrets configured
#   2. A local .env file (notebooks/.env) — for local / non-Colab runs
#   3. Existing os.environ (if the user exported them in the shell)
# Accepted keys: DEEPSEEK_API_KEY, DASHSCOPE_API_KEY, OPENAI_API_KEY.
# .env format: KEY=VALUE per line, '#' comments OK, optional surrounding quotes.
# The .env file is listed in .gitignore so it never ships to the repo.

_KEYS = ('DEEPSEEK_API_KEY', 'DASHSCOPE_API_KEY', 'OPENAI_API_KEY')

# 1) Colab Secrets (silently skipped off-Colab).
try:
    from google.colab import userdata
    for name in _KEYS:
        try:
            v = userdata.get(name)
            if v and not os.environ.get(name):
                os.environ[name] = v
        except Exception:
            pass
except Exception:
    pass

# 2) notebooks/.env — try a few plausible locations so this works whether the
#    kernel cwd is the repo root, the notebooks/ folder, or the Colab clone path.
_env_candidates = [
    Path.cwd() / '.env',
    Path.cwd() / 'notebooks' / '.env',
    Path('/content/Distorted-OCR-Compression/notebooks/.env'),
]
_env_loaded_from = None
for _p in _env_candidates:
    if _p.is_file():
        for line in _p.read_text().splitlines():
            line = line.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            k, v = line.split('=', 1)
            k = k.strip()
            v = v.strip().strip('"').strip("'")
            if k in _KEYS and v and not os.environ.get(k):
                os.environ[k] = v
        _env_loaded_from = str(_p)
        break

if _env_loaded_from:
    print(f'loaded .env from {_env_loaded_from}')
else:
    print('no .env found (fine if keys came from Colab Secrets or the shell)')

assert any(os.environ.get(k) for k in _KEYS), \
    '请至少设置 DEEPSEEK_API_KEY / DASHSCOPE_API_KEY / OPENAI_API_KEY 之一（Colab Secrets 或 notebooks/.env）'
print('keys present:', [k for k in _KEYS if os.environ.get(k)])

!nvidia-smi | head -6 || echo 'WARNING: 未检测到 GPU — DeepSeek-OCR 需要 A100/L4/A10G'


no .env found (fine if keys came from Colab Secrets or the shell)


AssertionError: 请至少设置 DEEPSEEK_API_KEY / DASHSCOPE_API_KEY / OPENAI_API_KEY 之一（Colab Secrets 或 notebooks/.env）

In [ ]:
%%bash
set -e
pip -q install 'transformers==4.46.3' 'tokenizers==0.20.3' 'accelerate>=0.34,<1.1' einops addict easydict
pip -q install Pillow matplotlib seaborn pandas requests tqdm
# flash-attn 不必安装：torch SDPA 已在 Ampere/Ada/Hopper 自动分发 FlashAttention-2 kernel。

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 163.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 124.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 55.4 MB/s eta 0:00:00


In [ ]:
import os, sys, shutil
from pathlib import Path

# Colab vs local bootstrapping.
# - Colab: clone the repo into /content, cd into it, expose it on sys.path.
# - Local (VS Code / Jupyter Lab / CLI): assume the notebook already lives
#   inside the repo; just resolve the repo root (the folder containing
#   `deepseek_pipeline/`) and add it to sys.path so `from deepseek_pipeline
#   import ...` works no matter which folder the kernel was launched from.
IS_COLAB = 'google.colab' in sys.modules or Path('/content').exists() and os.environ.get('COLAB_RELEASE_TAG')

if IS_COLAB:
    REPO_ROOT = Path('/content/Distorted-OCR-Compression')
    os.chdir('/content')
    if not REPO_ROOT.exists():
        os.system('git clone https://github.com/KuiyiGao/Distorted-OCR-Compression.git')
    else:
        os.system(f'cd {REPO_ROOT} && git pull')
    os.chdir(REPO_ROOT)
    WORKDIR = Path('/content')
else:
    # Walk up from the notebook's cwd looking for the repo marker.
    _here = Path.cwd().resolve()
    for _cand in (_here, *_here.parents):
        if (_cand / 'deepseek_pipeline').is_dir() and (_cand / 'render').is_dir():
            REPO_ROOT = _cand
            break
    else:
        raise RuntimeError(f'Could not find repo root (looking for deepseek_pipeline/ + render/) from {_here}')
    os.chdir(REPO_ROOT)
    WORKDIR = REPO_ROOT / 'runs'
    WORKDIR.mkdir(exist_ok=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f'IS_COLAB={IS_COLAB}  REPO_ROOT={REPO_ROOT}  WORKDIR={WORKDIR}')


## 1. 加载 CUADv1 并按文档切分（与上一版同种子，保证测试合同一致）

**采样策略**：每份测试合同中分别随机抽 `K_ANS` 个 gold 非空的问题 + `K_UNANS` 个 gold 为空的问题。
这一步是本 notebook 与上一版最大的区别——它把可答/不可答样本的比例从「随缘」固定为「可控」，从而让 EM/F1 不再被 abstention floor 吞没。

In [ ]:
import json, random, urllib.request, zipfile, io
from pathlib import Path

random.seed(0)  # 与上一版一致：前 10 份合同依然是测试集
N_TEST_DOCS = 20          # 扩大到 20 份（上一版 10 份）
K_ANS       = 2            # 每份合同抽 2 个可答问题
K_UNANS     = 2            # 每份合同抽 2 个不可答问题

CUAD_PATH = Path('render/data/CUADv1.json')
if not CUAD_PATH.exists():
    CUAD_PATH.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen('https://raw.githubusercontent.com/TheAtticusProject/cuad/main/data.zip') as r:
        with zipfile.ZipFile(io.BytesIO(r.read())) as z:
            with z.open('CUADv1.json') as src, open(CUAD_PATH, 'wb') as dst:
                dst.write(src.read())
cuad = json.load(open(CUAD_PATH))['data']
random.shuffle(cuad)

test_docs  = cuad[:N_TEST_DOCS]
train_docs = cuad[N_TEST_DOCS:]

train_contexts = [p['context'] for doc in train_docs for p in doc['paragraphs']]

test_items = []
rng = random.Random(1)
for doc in test_docs:
    for para in doc['paragraphs']:
        ctx = para['context']
        ans_qs = [q for q in para['qas'] if q.get('answers')]
        un_qs  = [q for q in para['qas'] if not q.get('answers')]
        rng.shuffle(ans_qs); rng.shuffle(un_qs)
        for qa in ans_qs[:K_ANS]:
            test_items.append({'qa_id': qa['id'], 'question': qa['question'],
                               'context': ctx, 'gold': [a['text'] for a in qa['answers']],
                               'answerable': True})
        for qa in un_qs[:K_UNANS]:
            test_items.append({'qa_id': qa['id'], 'question': qa['question'],
                               'context': ctx, 'gold': [],
                               'answerable': False})

n_ans   = sum(1 for t in test_items if t['answerable'])
n_unans = len(test_items) - n_ans
print(f'train: {len(train_docs)} docs -> {len(train_contexts)} contexts')
print(f'test : {len(test_docs)} docs -> {len(test_items)} QAs  |  answerable={n_ans}, unanswerable={n_unans}')

train: 490 docs -> 490 contexts
test : 20 docs -> 80 QAs  |  answerable=40, unanswerable=40


## 2. 加载已训练的 MemSlot；若无 checkpoint 则快速复训

`MemSlotSaliency.load()` 只恢复 `MemSlotAttention` 的权重，不恢复 `MemSlotConfig`，所以这里必须用**与上一版一致**的 `MemSlotConfig(n_slots=32, max_length=512, stride=256)`，否则权重形状对不上。

In [ ]:
from deepseek_pipeline import MemSlotSaliency, MemSlotConfig

cfg = MemSlotConfig(n_slots=32, epochs=2, batch_size=8, max_length=512, stride=256)
memslot = MemSlotSaliency(backbone_name='roberta-base', config=cfg)

CKPT = str(WORKDIR / 'memslot.pt')
if os.path.exists(CKPT):
    memslot.load(CKPT)
    print(f'loaded trained MemSlot from {CKPT}')
else:
    print('checkpoint not found, retraining (2 epochs on 200 train contracts) …')
    memslot.train_on_contracts(train_contexts[:200])
    memslot.save(CKPT)
    print(f'saved {CKPT}')


## 3. 统一 token 预算：为每条上下文定义 5× 与 10× 目标

我们用 `memslot.tokenizer`（RoBERTa）统一**衡量** token 数——这只是「同一把尺子量所有方法」，与下游 reader 用的 tokenizer 无关；目的是让 5× / 10× 在不同方法之间数值可比。

- 原始 tokens `N`：`memslot.tokenizer.encode(context, add_special_tokens=False)` 的长度。
- **目标预算** `B = N / R`，R ∈ {5, 10}。
- **Prune**：`keep_ratio = 1/R`；因为 prune 按「词」保留，保留后再用 RoBERTa 重新分词会有 ±10% 的偏差，我们记录实际 ratio。
- **Summary**：`target_tokens = B`。
- **OCR**：在 `{tiny:64, small:100, base:256, large:400}` 中选择 `argmin_mode |n_vis(mode) - B|`，记录达成 ratio。

In [ ]:
from deepseek_pipeline.metrics import RESOLUTION_MODES, vision_token_count_for_mode

OCR_MODE_CANDIDATES = ['tiny', 'small', 'base', 'large']   # 64, 100, 256, 400

def pick_ocr_mode_for_budget(budget_tokens: int) -> str:
    return min(OCR_MODE_CANDIDATES,
               key=lambda m: abs(RESOLUTION_MODES[m]['vision_tokens'] - budget_tokens))

TARGET_RATIOS = [5, 10]

# 预计算每条 test item 的原始 token 数和两档目标预算
for s in test_items:
    s['n_orig_tokens'] = len(memslot.tokenizer.encode(s['context'], add_special_tokens=False))
    s['budget'] = {R: max(1, s['n_orig_tokens'] // R) for R in TARGET_RATIOS}
    s['ocr_mode'] = {R: pick_ocr_mode_for_budget(s['budget'][R]) for R in TARGET_RATIOS}

import pandas as pd
budget_stats = pd.DataFrame([{ 'n_orig': s['n_orig_tokens'],
                               'budget_5x': s['budget'][5], 'mode_5x': s['ocr_mode'][5],
                               'budget_10x': s['budget'][10], 'mode_10x': s['ocr_mode'][10]} for s in test_items])
budget_stats.describe().round(1)

Token indices sequence length is longer than the specified maximum sequence length for this model (16243 > 512). Running this sequence through the model will result in indexing errors


,n_orig,budget_5x,budget_10x
count,80.0,80.0,80.0
mean,11266.8,2252.8,1126.2
std,9360.6,1872.0,936.0
min,394.0,78.0,39.0
25%,4043.0,808.0,404.0
50%,9251.5,1849.5,924.5
75%,14582.5,2915.8,1457.5
max,36113.0,7222.0,3611.0


## 4. 用已训练的 MemSlot 渲染每条测试合同的 TSVR 图像

渲染是**与问题无关**的（`render_tsvr_image` 只接收 `word_weights`）；同一张图既服务 5× 也服务 10×（OCR 在推理时选择不同分辨率档位来落到不同视觉 token 预算）。

In [ ]:
import os, hashlib
from render.render import render_tsvr_page

# Previous `render_tsvr_image` produced a fixed-width canvas whose height grew
# linearly with word count (~1024×30000 px for an 11k-token contract, up to
# 1024×95000 px on the 36k-token outliers). DeepSeek-OCR resizes input to a
# ~1024² patch grid, so those 1:30–1:90 scrolls were crushed into illegible
# smears — which is why `ocr-5x/10x` decoded empty text in the previous run
# (ratio_mean ≈ N_orig because `_text_tok_len` fell back to max(1, ...)).
#
# Fix: saliency-budgeted single-page render. For each compression ratio R we
# keep the top-B salient words (B = N/R, reading-order preserved) and lay
# them on a fixed A4-shaped page (1024×1448 px) with a font size picked by
# binary search so all B words fit. Two images per contract — one per R,
# because the kept word-set differs.

RENDERS_DIR = WORKDIR / 'renders_5x10x'
RENDERS_DIR.mkdir(parents=True, exist_ok=True)
page_cache, ww_cache = {}, {}

for s in test_items:
    ctx_key = hashlib.md5(s['context'].encode()).hexdigest()[:16]
    if ctx_key not in ww_cache:
        ww_cache[ctx_key] = memslot.word_weights(s['context'])
    s['word_weights'] = ww_cache[ctx_key]

    s['image'] = {}
    for R in TARGET_RATIOS:
        k = (ctx_key, R)
        if k not in page_cache:
            img = render_tsvr_page(s['word_weights'],
                                   n_words_budget=s['budget'][R],
                                   image_width=1024, page_aspect=1.414)
            path = str(RENDERS_DIR / f'{ctx_key}_r{R}.png')
            img.save(path)
            page_cache[k] = path
        s['image'][R] = page_cache[k]

# Preview: 5× and 10× pages for the first test item side-by-side.
from PIL import Image
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 9))
for ax, R in zip(axes, TARGET_RATIOS):
    im = Image.open(test_items[0]['image'][R])
    ax.imshow(im)
    ax.set_title(f'TSVR page — {R}× budget ({test_items[0]["budget"][R]} words, {im.size[0]}×{im.size[1]}px)')
    ax.axis('off')
plt.tight_layout(); plt.show()


## 5. 执行三路压缩（每种方法在 5× 和 10× 各跑一次）

- OCR：在同一张渲染图上分别用 `pick_ocr_mode_for_budget(N/5)` 与 `pick_ocr_mode_for_budget(N/10)` 推理。
- Prune：`keep_ratio = 0.20` 对应 5×，`0.10` 对应 10×。
- Summary：`target_tokens = N/5` 或 `N/10`。
- Full-context：保留原文，作为性能天花板。

In [ ]:
from deepseek_pipeline import DeepSeekOCRCompressor, SaliencyPruner, ApiSummarizer

# summary 基线：优先 DeepSeek，其次 Qwen；与 reader 的选择彼此独立
_SUM_PROVIDER = 'deepseek' if os.environ.get('DEEPSEEK_API_KEY') else 'qwen'

ocr        = DeepSeekOCRCompressor()
pruner     = SaliencyPruner(memslot.tokenizer)
summarizer = ApiSummarizer(memslot.tokenizer, provider=_SUM_PROVIDER)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

modeling_deepseekocr.py: 0.00B [00:00, ?B/s]

...

modeling_deepseekv2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/deepseek-ai/DeepSeek-OCR:
- modeling_deepseekv2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/deepseek-ai/DeepSeek-OCR:
- modeling_deepseekocr.py
- configuration_deepseek_v2.py
- conversation.py
- deepencoder.py
- modeling_deepseekv2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-000001.safetensors:   0%|          | 0.00/6.67G [00:00<?, ?B/s]

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at deepseek-ai/DeepSeek-OCR and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from tqdm.auto import tqdm

def _text_tok_len(txt: str) -> int:
    if not txt:
        return 1
    return max(1, len(memslot.tokenizer.encode(txt, add_special_tokens=False)))

all_runs = []
ocr_cache, summary_cache = {}, {}

for s in tqdm(test_items, desc='compress'):
    ctx_key = hashlib.md5(s['context'].encode()).hexdigest()[:16]
    N = s['n_orig_tokens']

    for R in TARGET_RATIOS:
        B = s['budget'][R]

        # ---- OCR: uses the ratio-specific saliency-budgeted page ----
        mode = s['ocr_mode'][R]
        img_path = s['image'][R]        # per-ratio page (NEW)
        k = (ctx_key, 'ocr', R, mode)
        if k not in ocr_cache:
            ocr_cache[k] = ocr.compress(img_path, mode=mode)
        r = ocr_cache[k]
        n_text = _text_tok_len(r.decoded_text)
        all_runs.append({'qa_id': s['qa_id'], 'question': s['question'], 'gold': s['gold'],
                         'answerable': s['answerable'],
                         'method': 'ocr', 'target_ratio': R, 'ocr_mode': mode,
                         'context': r.decoded_text,
                         'n_original_tokens': N,
                         'n_compressed_tokens': n_text,
                         'n_vision_tokens': r.n_vision_tokens})

        # ---- Prune ----
        keep = 1.0 / R
        r = pruner.compress(s['word_weights'], keep_ratio=keep)
        all_runs.append({'qa_id': s['qa_id'], 'question': s['question'], 'gold': s['gold'],
                         'answerable': s['answerable'],
                         'method': 'prune', 'target_ratio': R, 'keep_ratio': keep,
                         'context': r.text,
                         'n_original_tokens': N, 'n_compressed_tokens': r.n_tokens})

        # ---- Summary ----
        k = (ctx_key, 'summary', R)
        if k not in summary_cache:
            try:
                summary_cache[k] = summarizer.compress(s['context'], target_tokens=B, question=None)
            except Exception as e:
                print('summary err:', e); summary_cache[k] = None
        r = summary_cache[k]
        if r is not None:
            all_runs.append({'qa_id': s['qa_id'], 'question': s['question'], 'gold': s['gold'],
                             'answerable': s['answerable'],
                             'method': 'summary', 'target_ratio': R,
                             'context': r.text,
                             'n_original_tokens': N, 'n_compressed_tokens': r.n_tokens})

# Sanity check: if OCR decoded text is still mostly empty we have a rendering
# regression. Report so we notice fast (previous runs hid this behind silent
# fallbacks).
_ocr_empty = sum(1 for r in all_runs if r['method'] == 'ocr' and not r['context'].strip())
_ocr_total = sum(1 for r in all_runs if r['method'] == 'ocr')
print(f'OCR empty-decode rate: {_ocr_empty}/{_ocr_total}')

# full-context baseline (upper bound; fed to the same long-context reader)
for s in test_items:
    all_runs.append({'qa_id': s['qa_id'], 'question': s['question'], 'gold': s['gold'],
                     'answerable': s['answerable'],
                     'method': 'full', 'target_ratio': 1,
                     'context': s['context'],
                     'n_original_tokens': s['n_orig_tokens'], 'n_compressed_tokens': s['n_orig_tokens']})
print('total runs:', len(all_runs))


## 6. QA reader：长上下文优先（qwen-long / gpt-4o-mini）

`full` 基线的原文最长 36k tokens，`qwen-plus`（32k）会整片溢出返回 400。本 notebook 默认切换到 **`qwen-long`**（千万级上下文，与 `DASHSCOPE_API_KEY` 共用），并在开跑前做一次**预检调用**，确认 reader 在 `full` 上下文上能返回合法的 `{answer, confidence}`。
若要换成 **gpt-4o-mini**（128k 上下文），先运行下方 `OpenAIQAReader` 单元格再把 `reader = ...` 那一行替换为 `OpenAIQAReader(model='gpt-4o-mini')`，其它代码无需改动。


In [ ]:
# (可选) gpt-4o-mini reader — 与 ApiQAReader 接口一致
import os, json as _json, requests
from deepseek_pipeline.qa_eval import squad_em_f1, QAPrediction

class OpenAIQAReader:
    ENDPOINT = 'https://api.openai.com/v1/chat/completions'
    def __init__(self, model='gpt-4o-mini'):
        self.model = model
    def answer(self, context, question):
        api_key = os.environ['OPENAI_API_KEY']
        sys_msg = ('You are a legal-contract QA reader. Given a (possibly compressed) contract excerpt and a '
                   'question, reply with a JSON object of exactly two fields: '
                   '{"answer": <shortest exact-span answer or "" if unanswerable>, '
                   '"confidence": <integer 1-5 where 5 is certain>}. No commentary.')
        user_msg = f'Contract:\n{context}\n\nQuestion: {question}'
        resp = requests.post(self.ENDPOINT,
                             headers={'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'},
                             json={'model': self.model,
                                   'messages': [{'role':'system','content':sys_msg},{'role':'user','content':user_msg}],
                                   'temperature': 0.0, 'max_tokens': 256,
                                   'response_format': {'type': 'json_object'}},
                             timeout=120)
        resp.raise_for_status()
        raw = resp.json()['choices'][0]['message']['content'].strip()
        try:
            parsed = _json.loads(raw)
            ans = str(parsed.get('answer','')).strip()
            conf = float(parsed.get('confidence', 3)) / 5.0
        except Exception:
            ans, conf = raw, 0.5
        return ans, max(0.0, min(1.0, conf))
    def score(self, context, question, gold_answers):
        pred, conf = self.answer(context, question)
        em, f1 = squad_em_f1(pred, gold_answers)
        out = QAPrediction(question=question, prediction=pred, gold_answers=gold_answers, em=em, f1=f1)
        out.confidence = conf
        return out

In [ ]:
# Monkey-patch qwen-long support until the fix is pushed upstream.
from deepseek_pipeline import qa_eval as _qe

_orig_init = _qe.ApiQAReader.__init__
def _patched_init(self, provider='deepseek', model_name=None):
    if provider in ('qwen-long', 'qwen-max', 'qwen-turbo'):
        self.provider = provider
        self.api_key_env = 'DASHSCOPE_API_KEY'
        self.endpoint = _qe.ApiQAReader.QWEN_ENDPOINT
        self.model_name = model_name or provider
    else:
        _orig_init(self, provider)
_qe.ApiQAReader.__init__ = _patched_init
print('patched: qwen-long provider available')

patched: qwen-long provider available


In [ ]:
import os, requests
# Liveness check: confirm DASHSCOPE_API_KEY works before the main loop.
assert os.environ.get('DASHSCOPE_API_KEY'), 'DASHSCOPE_API_KEY 未设置（放到 Colab Secrets 再运行本单元格）'

r = requests.post(
    'https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions',
    headers={'Authorization': f'Bearer {os.environ["DASHSCOPE_API_KEY"]}',
             'Content-Type': 'application/json'},
    json={'model': 'qwen-turbo',
          'messages': [{'role': 'user', 'content': 'ping'}],
          'max_tokens': 5},
    timeout=30)
print('status:', r.status_code)
print('body  :', r.text[:400])


In [ ]:
from deepseek_pipeline import ApiQAReader

# Prefer qwen-long (10M-token context). Note: we MUST pass model_name='qwen-long'.
# Previous bug: model_name='qwen-turbo' silently overrode the provider, capping the
# context at ~8k. That made the 36k `full` baseline get truncated to just the
# document header — which is why preflight echoed the first paragraph verbatim and
# every method tied at EM=0. Passing the correct model fixes the whole experiment.
if os.environ.get('DASHSCOPE_API_KEY'):
    reader = ApiQAReader(provider='qwen-long', model_name='qwen-long')
elif os.environ.get('OPENAI_API_KEY'):
    reader = OpenAIQAReader(model='gpt-4o-mini')
else:
    reader = ApiQAReader(provider='deepseek')
    print('WARN: using deepseek-chat (context ~64k); `full` baseline may overflow.')
print('QA reader:', reader.__class__.__name__, '|',
      getattr(reader, 'model_name', getattr(reader, 'model', '?')))

# ── Pre-flight: sanity-check the reader against the full (longest) context so we
# fail fast instead of after 556 silent errors. Also guard against the "reader
# echoes the header" failure mode that truncation produces. ───────────────────
_probe = max(test_items, key=lambda s: s['n_orig_tokens'])
print(f'preflight: probing reader with doc of {_probe["n_orig_tokens"]} orig tokens …')
try:
    _res = reader.score(_probe['context'], _probe['question'], _probe['gold'])
    print(f'  ok  | prediction={_res.prediction!r}  em={_res.em:.2f}  f1={_res.f1:.2f}  conf={_res.confidence:.2f}')
    # Heuristic: if the prediction is a verbatim prefix of the context, the reader
    # is almost certainly truncating and parroting the header — abort.
    if _res.prediction and _probe['context'].lstrip().startswith(_res.prediction[:80]):
        raise RuntimeError('reader returned a verbatim prefix of the context — '
                           'likely silent truncation. Check model_name / context limit.')
except Exception as e:
    raise RuntimeError(f'QA reader preflight failed — aborting before the main loop.\n{type(e).__name__}: {e}')

# ── Main QA loop: keep the call failure mode visible (status + error_msg). ────
for run in all_runs:
    run.setdefault('prediction', ''); run.setdefault('em', float('nan'))
    run.setdefault('f1', float('nan')); run.setdefault('confidence', float('nan'))
    run.setdefault('status', 'pending'); run.setdefault('error_msg', '')

for run in tqdm(all_runs, desc='QA'):
    try:
        res = reader.score(run['context'], run['question'], run['gold'])
        run['prediction'] = res.prediction
        run['em'], run['f1'], run['confidence'] = res.em, res.f1, res.confidence
        run['status'] = 'ok'
    except Exception as e:
        run['prediction'] = ''
        run['em'] = run['f1'] = run['confidence'] = float('nan')
        run['status'] = 'api_error' if 'HTTP' in type(e).__name__ or 'requests' in str(type(e)) else 'error'
        run['error_msg'] = f'{type(e).__name__}: {e}'

from collections import Counter
print('status counts:', dict(Counter(r['status'] for r in all_runs)))


QA reader: ApiQAReader | qwen-long
preflight: probing reader with doc of 36113 orig tokens …
  ok  | prediction='as of the 16th day of February, 2000 ("Effective Date")'  em=0.00  f1=0.71  conf=1.00


QA:   0%|          | 0/556 [00:00<?, ?it/s]

status counts: {'ok': 556}


## 7. 评测：诚实分母 + 区分失败与弃答

- `success_rate`：该 label 下 `status=='ok'` 的比例——reader 是否真的在工作；
- `EM_ans / F1_ans`：仅在成功且可答的样本上算 SQuAD EM/F1；
- `abstain_acc_clean`：仅在成功且不可答的样本上算 abstention 准确率；区别于旧的 `abstain_acc`（把失败也算成弃答）；
- `AUPR / P@80%R`：按 CUAD 官方口径，只喂成功样本的置信度；
- `ratio_mean`：所有方法统一用「文本 tokens」作分母计算的实际压缩比。


In [ ]:
import pandas as pd, numpy as np
from deepseek_pipeline import cuad_evaluate

df = pd.DataFrame(all_runs)
df['compression_ratio'] = df['n_original_tokens'] / df['n_compressed_tokens'].clip(lower=1)
df['label'] = df.apply(lambda r: f"{r['method']}-{r['target_ratio']}x" if r['method'] != 'full' else 'full', axis=1)
df.to_csv(WORKDIR / 'cuad_5x10x_results.csv', index=False)

def _safe_mean(s):
    s = pd.Series(s).dropna()
    return float(s.mean()) if len(s) else float('nan')

rows = []
for label, sub in df.groupby('label'):
    ok  = sub[sub['status'] == 'ok']
    ans = ok[ok['answerable']]
    una = ok[~ok['answerable']]
    if len(ok):
        s_full = cuad_evaluate(ok['prediction'].tolist(), ok['gold'].tolist(),
                               confidences=ok['confidence'].tolist())
        aupr, p80 = s_full.aupr, s_full.precision_at_80_recall
    else:
        aupr = p80 = float('nan')
    rows.append({
        'label': label,
        'success_rate': float((sub['status'] == 'ok').mean()),
        'EM_ans':  _safe_mean(ans['em']),
        'F1_ans':  _safe_mean(ans['f1']),
        'abstain_acc_clean': float((una['prediction'].str.strip() == '').mean()) if len(una) else float('nan'),
        'AUPR':  aupr,
        'P@80%R': p80,
        'ratio_mean': _safe_mean(sub['compression_ratio']),
        'n_ok_ans': len(ans), 'n_ok_unans': len(una),
        'n_total':  len(sub),
    })
summary = pd.DataFrame(rows).sort_values('label').set_index('label').round(3)

# Gate: if `full` is not mostly healthy, the experiment is not valid.
_full_sr = summary.loc['full', 'success_rate'] if 'full' in summary.index else 0.0
if _full_sr < 0.9:
    print(f'[WARN] full-context success_rate = {_full_sr:.2f} < 0.90 — reader appears unhealthy; '
          'inspect df[df.status!="ok"][["label","error_msg"]].head() before trusting the table below.')
summary


## 8. 可视化

- **左图**：`F1_ans` — 每个方法在可答子集上的真实 F1；
- **中左**：`abstain_acc_clean` — 成功样本里不可答题的弃答准确率；
- **中右**：`success_rate` — 接口健康度（低于 0.9 的条形应被视为不可信）；
- **右图**：`ratio_mean` — 统一用文本-token 口径计算的实际压缩比。


In [ ]:
import matplotlib.pyplot as plt, seaborn as sns, pandas as pd
sns.set_theme(style='whitegrid', context='talk')

plot_df = summary.reset_index().copy()
plot_df['method'] = plot_df['label'].str.replace('-5x','').str.replace('-10x','')
plot_df['ratio']  = plot_df['label'].map(lambda x: 'full' if x == 'full' else x.split('-')[-1])

metrics = [('F1_ans', 'F1 on answerable'),
           ('abstain_acc_clean', 'Abstention accuracy (clean)'),
           ('success_rate', 'API success rate'),
           ('ratio_mean', 'Achieved compression ratio')]

fig, axes = plt.subplots(2, 2, figsize=(18, 11))
for ax, (metric, title) in zip(axes.ravel(), metrics):
    sns.barplot(data=plot_df, x='method', y=metric, hue='ratio', ax=ax,
                palette={'5x':'#1f77b4','10x':'#d62728','full':'#7f7f7f'})
    ax.set_title(title); ax.set_xlabel(''); ax.set_ylabel(metric)
plt.suptitle(f'OCR vs Prune vs Summary — 5× / 10× | reader={reader.__class__.__name__}',
             fontsize=16, fontweight='bold')
plt.tight_layout(); plt.savefig(WORKDIR / '5x10x_compare.png', dpi=150); plt.show()


In [ ]:
# Per-QA F1 heatmap (answerable subset, successful API calls only).
df_ok = df[(df['status'] == 'ok') & df['answerable']].copy()
if len(df_ok):
    pivot = df_ok.pivot_table(index='qa_id', columns='label', values='f1')
    fig, ax = plt.subplots(figsize=(14, max(3, 0.45 * len(pivot))))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1,
                cbar_kws={'label': 'F1'}, ax=ax)
    ax.set_title('Per-QA F1 (answerable subset, status=ok) — 5× / 10× compression')
    ax.set_xlabel(''); ax.set_ylabel('CUAD QA id')
    plt.tight_layout(); plt.savefig(WORKDIR / '5x10x_heatmap.png', dpi=150); plt.show()
else:
    print('No successful answerable runs — heatmap skipped. Inspect df[df.status != "ok"] for failures.')


## 9. 实验设计声明（学术口径）

**研究问题**。在相同的压缩比（5× 和 10×）下，三种压缩路径——(i) DeepSeek-OCR 视觉 token 压缩、(ii) 基于 MemSlot 显著性的词级剪枝、(iii) API 摘要——对 CUAD 合同 QA 任务的保真度有何差异？

**数据划分**。按**合同**而不是按 QA 切分：`random.seed(0)` 下前 20 份合同为测试集，其余 490 份为训练集。MemSlot 的无监督训练只见过训练集中的 200 份合同，测试合同对模型是完全未见的。

**无问题泄漏**。MemSlot 的目标函数是 *重建 + 槽多样性*，永远不接触问题、答案或条款类别；OCR 渲染和剪枝都基于这个 question-agnostic 的显著性分数；摘要基线在调用时也显式把 `question=None` 传下去。

**分层 QA 采样**。每份测试合同抽 `K_ANS=2` 个可答 + `K_UNANS=2` 个不可答问题，合计 80 条 QA。这个比例远比 CUAD 自身的 ~15% 可答率要均衡，目的是让 EM/F1 不再被 abstention floor 吞没（即上一版 notebook 出现的病灶）。

**统一压缩预算**。以 `memslot.tokenizer`（RoBERTa）为统一度量，对每条上下文定义原始 token 数 `N` 与目标预算 `B = N/R`（`R ∈ {5,10}`）：
- Prune：`keep_ratio = 1/R`；
- Summary：`target_tokens = B`；
- OCR：在 `{tiny:64, small:100, base:256, large:400}` 四档中选 argmin `|n_vis - B|`；离散档位带来的偏差在 `ratio_mean` 列中如实报告。

**QA reader 固定且共享**。三种压缩路径都喂到同一个 API reader（默认 Qwen-plus，可换 gpt-4o-mini），`temperature=0`，`response_format=json_object`。这样唯一变量就是上下文压缩方式。

**评价指标**。
- **EM_ans / F1_ans**：仅在可答子集上计算 SQuAD-style 指标；是压缩信息损失的真正差异信号。
- **abstention accuracy**：不可答子集中模型返回空串的比例。过度压缩会让 reader 失去「无答」证据，被迫编造答案。
- **AUPR / P@80%R**：按 CUAD 官方口径，用 reader 自报的 1-5 置信度（归一化到 [0,1]）排序后计算 PR 曲线。
- **实际压缩比**：`N_orig / N_compressed`；OCR 分支的分母是视觉 token 数，prune/summary 是 RoBERTa 分词后的文本 token 数——这是为了让三者共享同一个「下游 decoder 需要消化几个 slot」的语义。

**局限**。
1. OCR 的视觉 token 档位离散，实际压缩比通常落在 5× 附近而非精确 5.00×；10× 可能走到 "small" 或 "tiny" 档位，使两个实验点之间不是严格等差。
2. QA reader 并非 DeepSeek3B 解码器本身，外部 API reader 看到的只是 OCR 解码后的文本——OCR 的视觉压缩优势在 *reader 的 attention 成本* 上不直接体现。我们报告视觉 token 比仅是为了指示 *上游* 压缩强度，比较 QA 保真度用的是同一个 reader。
3. 80 条 QA 仍偏小，AUPR/P@80%R 的置信区间较宽；若预算允许可把 `N_TEST_DOCS` 拉到 50 以上。

## 10. 产物

- `cuad_5x10x_results.csv` — 每次（method × ratio × qa_id）的 prediction/gold/em/f1/confidence/压缩比
- `5x10x_compare.png` — 方法对比柱图（F1 / abstention / ratio）
- `5x10x_heatmap.png` — 可答子集逐题 F1 热力图
- `memslot.pt` — 复用的 MemSlot 权重
- `/content/renders_5x10x/` — 每份测试合同的 TSVR 渲染图